# 12. OpenAI API — Responses 기본기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/12_openai_api.ipynb)

> 🧭 **스토리** — 오후 내내 로컬 모델로 버텼다. 이제 회사가 API 키를 내줬다. 규정 도우미를 서비스로 만들기 전에 **비용·속도·형식·오류**를 손에 익힌다.
>
> 📖 **이론 연결** — 이론1 Part 2(토큰·누적 대화), Part 4(추론·effort), Part 5(프롬프트 캐싱), Part 6(API 옵션·응답 객체·구조화 출력) · ⏱ **15분** · 🖥 **CPU**
>
> GPU가 필요 없습니다. 기본 런타임(CPU)으로 실행하세요.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
이 노트북은 GPU가 필요 없습니다. OpenAI SDK와 Pydantic만 씁니다. 키 셀에서 수업용 키를 붙여 넣으세요. 키를 코드에 적거나 출력하지 마세요.

In [ ]:
%pip -q install openai pydantic pandas

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
# 입력한 키는 모델 조회 API(GET /v1/models/{모델} = client.models.retrieve)로 바로 확인합니다. 토큰을 쓰지 않아 비용이 없습니다.
import os, json, urllib.request, urllib.error
from getpass import getpass
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.

def check_key(key: str):
    """키 확인 → (상태, 안내). 키 문자열과 오류 본문은 출력하지 않는다(유형·상태 코드만)."""
    req = urllib.request.Request(f"https://api.openai.com/v1/models/{MODEL_API}",
                                 headers={"Authorization": f"Bearer {key}"})
    try:
        with urllib.request.urlopen(req, timeout=15):
            return "ok", "키 확인 완료"
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode()).get("error", {})        # 판단에만 쓰고 출력하지 않는다
        except Exception:
            err = {}
        code, msg = err.get("code"), str(err.get("message", "")).lower()
        scope = "insufficient permissions" in msg or "missing scopes" in msg   # 권한 범위가 좁은 제한 키
        if e.code == 401 and not scope and code in ("invalid_api_key", None):
            return "bad", "키가 올바르지 않습니다(401). 빠진 글자·앞뒤 공백 없이 다시 붙여 넣으세요."
        if e.code in (401, 403):
            return "ok", f"키는 유효하지만 모델 조회 권한이 없습니다({e.code}). 실습은 진행하고, 호출이 막히면 강사에게 알려 주세요."
        if e.code == 404:
            return "ok", f"키는 유효하지만 모델 {MODEL_API}에 접근할 수 없습니다(404). 강사에게 알려 주세요."
        if e.code == 429:
            return "ok", "키는 유효합니다. 요청 한도에 걸렸습니다(429). 잠시 뒤 다시 실행하세요."
        return "err", f"확인 실패: HTTPError {e.code}. 잠시 뒤 이 셀을 다시 실행하세요."
    except Exception as e:
        return "err", f"확인 실패: {type(e).__name__}. 네트워크 연결을 확인하고 이 셀을 다시 실행하세요."

key = os.environ.get("OPENAI_API_KEY", "").strip()
for attempt in range(1, 4):
    if not key:
        key = getpass(f"OpenAI API 키 ({attempt}/3): ").strip()
    status, msg = check_key(key)
    print(msg)
    if status != "bad":
        break
    key = ""                                   # 틀린 키는 버리고 다시 입력받는다
if status == "bad":
    os.environ.pop("OPENAI_API_KEY", None)
    raise RuntimeError("키 확인 3회 실패: 강사에게 키를 다시 받은 뒤 이 셀을 다시 실행하세요.")
os.environ["OPENAI_API_KEY"] = key
print("모델:", MODEL_API)

In [ ]:
# ⚙️ 비용 계산 — 2026-10 기준 공식 단가(USD per 1M tokens). https://developers.openai.com/api/docs/pricing
PRICES = {  # (입력, 캐시 입력, 출력)
    "gpt-6-luna":   (0.10, 0.01, 0.50),
    "gpt-5-nano":   (0.05, 0.005, 0.40),
    "gpt-5.6-luna": (0.20, 0.02, 1.20),
    "gpt-6-astra":  (10.0, 1.0, 50.0),
    "text-embedding-3-small": (0.02, 0.02, 0.0),
}
CACHE_WRITE_X = {"gpt-6-luna": 1.25, "gpt-5.6-luna": 1.25, "gpt-6-astra": 1.25}   # GPT-5.6 이후 모델: 캐시 쓰기 = 입력 단가 × 1.25
def cost_usd(model: str, in_tok: int, out_tok: int = 0, cached_tok: int = 0, write_tok: int = 0) -> float:
    """in_tok = usage.input_tokens(캐시 읽기 cached_tok · 캐시 쓰기 write_tok 포함 전체 입력)."""
    p_in, p_cache, p_out = PRICES.get(model, (0.10, 0.01, 0.50))
    p_write = p_in * CACHE_WRITE_X.get(model, 1.0)
    return ((in_tok - cached_tok - write_tok) * p_in + write_tok * p_write + cached_tok * p_cache + out_tok * p_out) / 1_000_000
def cost_krw(usd: float, rate: float = 1400.0) -> float:
    return usd * rate

In [ ]:
# ⚙️ API 오류를 한국어로 설명하는 헬퍼 — 8절에서 자세히 봅니다. 키가 없어도 노트북이 멈추지 않게 합니다.
import openai, time, json
from openai import OpenAI
client = OpenAI()
def explain_error(e):
    if isinstance(e, openai.AuthenticationError):
        print("🔑 401 인증 실패 — 키가 없거나 틀렸거나 폐기되었습니다. 키 셀을 다시 실행하세요.")
    elif isinstance(e, openai.RateLimitError):
        if "project_spend_limit_exceeded" in str(e):
            print("💸 429 project_spend_limit_exceeded — 수업용 프로젝트의 월 지출 하드 한도에 도달했습니다. "
                  "기다려도 풀리지 않습니다. 강사에게 알려 주세요.")
        else:
            print("⏳ 429 속도 제한 — 분당 요청·토큰 한도를 넘었습니다. 몇 초 뒤 다시 실행하세요.")
    elif isinstance(e, openai.APIConnectionError):
        print("🌐 연결 실패 — 네트워크를 확인하세요.")
    elif isinstance(e, openai.APIError) and "rate limit" in str(e).lower():   # 스트리밍 중 속도 제한은 APIError로 온다
        print("⏳ 429 속도 제한(스트리밍 중) — 몇 초 뒤 다시 실행하세요.")
    else:
        print("⚠️", type(e).__name__, getattr(e, "status_code", "") or "")   # 오류 본문은 출력하지 않습니다

## 1. 가장 짧은 호출 🔵
새 프로젝트는 **Responses API**(`client.responses.create`)를 씁니다. 결과 글은 `r.output_text`, 토큰 사용량은 `r.usage`에 있습니다.
`reasoning={"effort": "none"}`은 "생각 단계 없이 바로 답하라"는 뜻입니다. 단순 질문에는 이것이 가장 싸고 빠릅니다.

In [ ]:
try:
    r = client.responses.create(model=MODEL_API, input="한빛테크 신입사원에게 건네는 환영 인사를 한 문장으로 써 줘.",
                                reasoning={"effort": "none"}, max_output_tokens=80)
    print(r.output_text)
    print(r.usage)
except Exception as e:
    explain_error(e)

💡 **해석** — `usage`에 `input_tokens`, `output_tokens`, `input_tokens_details.cached_tokens`(캐시 읽기, 입력 단가의 0.1배), `input_tokens_details.cache_write_tokens`(캐시 쓰기, 1.25배), `output_tokens_details.reasoning_tokens`가 들어 있습니다. **과금은 이 숫자로 합니다.** 노트북 02에서 tiktoken으로 센 값은 추정일 뿐입니다.

## 2. 비용 함수 — `usage` × 단가 🔵
호출할 때마다 비용을 찍는 작은 래퍼를 만듭니다. 서비스 로그에 이 한 줄을 남기면 월말에 놀랄 일이 없습니다.

In [ ]:
def ask(input, instructions=None, effort="none", max_output_tokens=300, **kw):
    """Responses 호출 + 비용·시간 출력. 실패하면 None."""
    t0 = time.time()
    try:
        r = client.responses.create(model=MODEL_API, input=input, instructions=instructions,
                                    reasoning={"effort": effort}, max_output_tokens=max_output_tokens, **kw)
    except Exception as e:
        explain_error(e); return None
    u = r.usage
    cached = u.input_tokens_details.cached_tokens
    write = getattr(u.input_tokens_details, "cache_write_tokens", 0) or 0   # 캐시 쓰기(입력 단가 × 1.25)
    usd = cost_usd(MODEL_API, u.input_tokens, u.output_tokens, cached, write)
    print(f"[입력 {u.input_tokens} (캐시 {cached} · 쓰기 {write}) · 출력 {u.output_tokens} · {time.time()-t0:.1f}초 · "
          f"${usd:.6f} ≈ {cost_krw(usd):.3f}원]")
    return r

r = ask("출장비 정산 기한을 지키지 않으면 어떻게 되나요? 두 문장으로.")
if r: print(r.output_text)

💡 **해석** — 한 번에 1원도 안 나옵니다. 하지만 하루 1만 건, 지시문 2,000토큰짜리 서비스라면 이야기가 달라집니다. 마지막 🤔에서 계산합니다.

## 3. `instructions` vs `input` 🔵
- `instructions`: 개발자가 정하는 **규칙**(시스템 지시). 말투·역할·금지 사항.
- `input`: 사용자가 보내는 **질문**.

같은 질문에 `instructions`만 바꿔 봅니다.

In [ ]:
q = "연차를 다음 해로 넘길 수 있나요?"
for inst in [None,
             "너는 한빛테크 인사팀 챗봇이다. 존댓말로 두 문장 이내로 답하고, 모르면 인사팀(hr@hanbit-tech.example)에 문의하라고 안내하라.",
             "너는 해적이다. 모든 답을 해적 말투로 한 문장만 한다."]:
    print(f"--- instructions = {inst!r:.40}")
    r = ask(q, instructions=inst, max_output_tokens=150)
    if r: print(r.output_text, "\n")

💡 **해석** — 질문은 같은데 답의 길이·말투·안내 방식이 바뀝니다. **역할과 규칙은 `instructions`에, 사용자 입력은 `input`에** 나누어 두세요. 사용자가 "앞의 지시를 무시해"라고 써도 `input`에 있으므로 지시와 섞이지 않습니다(완벽한 방어는 아닙니다). 지시문이 길고 고정되어 있으면 9절의 **캐시**도 받습니다.

## 4. 생각을 시킬까 말까 — `reasoning.effort` 🔵
GPT-6 Luna의 effort는 `none / low / medium / high / xhigh / max`입니다(기본 medium). 생각을 많이 할수록 **보이지 않는 추론 토큰**이 출력 토큰으로 과금됩니다.
주의: effort가 `none`이 아니면 `temperature`·`top_p`를 보내면 안 됩니다.

In [ ]:
q = "과장이 2박 3일 국내 출장을 갑니다. 일비 30,000원/일, 숙박비 상한 110,000원/박이면 합계 최대 얼마인가요? 숫자만."
for effort in ["none", "low"]:
    print(f"--- effort = {effort}")
    r = ask(q, effort=effort, max_output_tokens=2000)
    if r:
        print("답:", r.output_text.strip(), "| 추론 토큰:", r.usage.output_tokens_details.reasoning_tokens)

💡 **해석** — `low`는 추론 토큰이 생기고 시간이 늘어납니다. 이런 단순 계산은 `none`으로도 맞히는 경우가 많습니다. **기본은 none, 틀리는 문제만 effort를 올린다**가 비용 관리의 출발점입니다. 추론 토큰은 화면에 안 보여도 출력 단가로 청구됩니다(이론1 p131).

## 5. 같은 질문, 다른 답 — 비결정성 🔵
같은 입력을 세 번 보냅니다.

In [ ]:
for i in range(3):
    r = ask("한빛테크 사내 메신저 이름을 하나 지어 줘. 이름만.", max_output_tokens=30)
    if r: print(f"{i+1}회:", r.output_text.strip())

💡 **해석** — 세 번이 모두 같을 수도, 서로 다를 수도 있습니다(코랩 실측 2026-10-17: 세 번 모두 같은 이름). LLM은 다음 토큰을 **확률로 뽑기** 때문에 한 후보의 확률이 아주 높으면 같은 답이 반복되고, 호출을 늘리면 다른 답이 섞입니다(이론1 p85~86). `effort="none"`일 때는 `temperature`를 낮춰 흔들림을 줄일 수 있지만, 0으로 해도 완전히 같다고 보장되지 않습니다. 그래서 **답의 형식은 스키마로 고정**하고(다음 절) 정확성은 평가 세트로 반복 측정합니다.

## 6. 구조화 출력 — Pydantic으로 형식 고정 🔵
사내 문의를 분류해 티켓 시스템에 넣으려면 답이 **항상 같은 JSON 형식**이어야 합니다. `responses.parse(text_format=모델클래스)`를 쓰면 서버가 스키마를 강제하고, 결과가 바로 파이썬 객체(`r.output_parsed`)로 옵니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Inquiry(BaseModel):
    category: Literal["출장", "휴가", "교육", "기타"]
    urgency: int = Field(description="긴급도 1(낮음)~5(매우 급함)")
    summary: str = Field(description="한 문장 요약")

CLS_INST = "한빛테크 인사팀으로 들어온 문의를 분류한다. category, urgency(1~5), summary를 채운다."
def classify(text):
    try:
        r = client.responses.parse(model=MODEL_API, instructions=CLS_INST, input=text, text_format=Inquiry,
                                   reasoning={"effort": "none"})
        return r.output_parsed
    except Exception as e:
        explain_error(e); return None

res = classify("내일 아침 9시 부산 출장인데 아직 전자결재 승인을 못 받았어요. 어떻게 하죠?")
if res:
    print(res)
    print(type(res).__name__, "→", res.category, res.urgency)
print(json.dumps(Inquiry.model_json_schema(), ensure_ascii=False)[:300], "…")

💡 **해석** — `category`는 네 값 중 하나만 나옵니다. "출장 관련" 같은 변형이나 따옴표 깨진 JSON이 나오지 않습니다. 노트북 05에서 로컬 1.2B 모델에게 프롬프트로 "JSON으로 답해"라고 **부탁**했을 때 몇 번 깨졌는지 떠올려 보세요. 부탁은 실패할 수 있습니다. 여기서는 서버가 디코딩 단계에서 스키마를 **강제**하므로 형식 오류가 사실상 0입니다. 다만 **내용**(긴급도가 적절한지)은 여전히 틀릴 수 있습니다. 형식과 정확성은 다른 문제입니다.

## 7. 스트리밍 — 한 글자씩 받기 🔵
챗봇 화면에서 답이 타자 치듯 나오는 것이 스트리밍입니다. 전체 시간은 비슷해도 **첫 글자가 빨리 보여서** 체감 속도가 좋아집니다.

In [ ]:
try:
    t0 = time.time(); first = None
    stream = client.responses.create(model=MODEL_API, input="출장 가기 전 챙길 것 5가지를 짧게 나열해 줘.",
                                     reasoning={"effort": "none"}, max_output_tokens=200, stream=True)
    for ev in stream:
        if ev.type == "response.output_text.delta":
            if first is None: first = time.time() - t0
            print(ev.delta, end="", flush=True)
    print(f"\n\n첫 글자까지 {first:.2f}초 · 전체 {time.time()-t0:.2f}초" if first else "")
except Exception as e:
    explain_error(e)

💡 **해석** — 스트림은 이벤트 여러 종류가 섞여 옵니다. 글자 조각은 `response.output_text.delta`이고, 끝을 알리는 `response.completed` 이벤트에 `usage`가 들어 있습니다. 첫 글자 시간(TTFT)과 전체 시간의 차이를 비교해 보세요.

## 8. 오류 처리 — 401과 429 🔵
수업용 키에는 **프로젝트 월 지출 하드 한도**가 걸려 있습니다. 한도를 넘으면 그 프로젝트의 모든 호출이 `429`를 받고, 오류 코드가 `project_spend_limit_exceeded`입니다. 같은 429라도 성격이 다릅니다.

| 상황 | 예외 | 대처 |
|---|---|---|
| 키가 틀림·폐기됨 | `openai.AuthenticationError`(401) | 키 확인. 재시도해도 소용없음 |
| 분당 요청·토큰 초과 | `openai.RateLimitError`(429) | 잠깐 기다렸다 재시도(지수 백오프) |
| 지출 하드 한도 초과 | `RateLimitError`(429, `project_spend_limit_exceeded`) | **기다려도 안 풀림.** 한도 상향 또는 다음 달 |

일부러 틀린 키로 401을 만들어 봅니다. 진짜 키는 건드리지 않습니다.

In [ ]:
bad = OpenAI(api_key="sk-wrong-key-for-demo")
try:
    bad.responses.create(model=MODEL_API, input="hi", reasoning={"effort": "none"}, max_output_tokens=16)
except openai.AuthenticationError as e:
    print("잡았다:", type(e).__name__, "| status", e.status_code)
    explain_error(e)
except Exception as e:
    explain_error(e)

def ask_with_retry(text, tries=3):
    """속도 제한(429)만 재시도. 지출 한도 429는 바로 포기합니다."""
    for i in range(tries):
        try:
            return client.responses.create(model=MODEL_API, input=text, reasoning={"effort": "none"}, max_output_tokens=50)
        except openai.RateLimitError as e:
            if "project_spend_limit_exceeded" in str(e):
                explain_error(e); return None
            wait = 2 ** i; print(f"429 → {wait}초 뒤 재시도"); time.sleep(wait)
        except Exception as e:
            explain_error(e); return None
r = ask_with_retry("안녕?")
if r: print("재시도 래퍼 성공:", r.output_text)

💡 **해석** — OpenAI SDK는 연결 오류와 429를 기본 2번 자동 재시도합니다(`max_retries`). 그래도 실패하면 예외가 올라옵니다. 실서비스에서는 **오류 종류별로 다르게** 처리하세요. 401은 사람에게 알리고, 일반 429는 기다렸다 재시도하고, 지출 한도 429는 즉시 멈추고 알림을 보냅니다(이론2 p146 키·비용 운영).

## 9. 프롬프트 캐싱 — 같은 앞부분은 10분의 1 🔵
요청의 **앞부분(접두부)**이 1,024토큰 이상이고 최근 요청과 같으면, 서버가 그 부분을 캐시에서 꺼내 씁니다. 캐시 입력 단가는 일반 입력의 **10분의 1**입니다.
규정집 전체(약 2,000토큰 이상)를 `instructions`에 넣고 질문만 바꿔 두 번 호출합니다.

In [ ]:
rules = load_text("hanbit_rules_gt.md")
RULES_INST = "너는 한빛테크 규정 도우미다. 아래 규정에 근거해서만 한 문장으로 답한다.\n\n" + rules
for q in ["과장의 국내 출장 일비는?", "본인 결혼 시 경조금은?"]:
    print("Q:", q)
    r = ask(q, instructions=RULES_INST, max_output_tokens=80, prompt_cache_key="hanbit-rules-v1")
    if r: print("A:", r.output_text.strip(), "\n")

💡 **해석** — 첫 호출은 `캐시 0`, 두 번째 호출은 `캐시`가 수천 토큰으로 찍히는 것이 정상입니다(접두부가 1,024토큰 이상이어야 캐시 대상). 그만큼 입력 비용이 10분의 1이 됩니다. 캐시는 **앞부분이 글자 하나까지 같아야** 맞습니다. 그래서 고정된 지시문·문서를 앞에, 바뀌는 질문을 뒤에 둡니다. `prompt_cache_key`는 고객·용도별로 캐시를 구분하는 선택 항목입니다(GPT-6에서는 적중을 위해 필수는 아님). 캐시는 마지막 사용 후 30분이 지나면 사라지므로(GPT-6 기준) 첫 호출에서 0이 나와도 이상한 일이 아닙니다.

### 9-2. 반복 업무·평가에서 캐싱: 같은 접두부로 질문 10개 🔵
평가셋 `hanbit_qa.json` 10문항을 **같은 규정집 접두부**로 돌립니다. 호출마다 `cached_tokens`를 기록하고, 캐시가 없었다면 얼마였을지 비교합니다. 평가 루프·배치 처리·에이전트 루프(툴 호출마다 같은 시스템 프롬프트)처럼 **같은 앞부분을 반복해서 보내는 일**이 캐싱의 주 무대입니다.

In [ ]:
import pandas as pd
qa = load_json("hanbit_qa.json")[:10]          # 1~10번: 규정집 안에 답이 있는 문항
rows, usd_with, usd_without = [], 0.0, 0.0
for item in qa:
    r = ask(item["q"], instructions=RULES_INST, max_output_tokens=60, prompt_cache_key="hanbit-rules-v1")
    if not r: continue
    u = r.usage; cached = u.input_tokens_details.cached_tokens
    write = getattr(u.input_tokens_details, "cache_write_tokens", 0) or 0
    usd_with    += cost_usd(MODEL_API, u.input_tokens, u.output_tokens, cached, write)   # 실제 과금(캐시 쓰기 1.25배 포함)
    usd_without += cost_usd(MODEL_API, u.input_tokens, u.output_tokens, 0)        # 캐시가 없었다면
    ok = all(k in r.output_text for k in item["keywords"])                       # 정답 키워드 포함 여부
    rows.append({"id": item["id"], "질문": item["q"][:20] + "…", "입력": u.input_tokens, "캐시": cached, "쓰기": write,
                 "출력": u.output_tokens, "정답": "O" if ok else "X", "답": r.output_text.strip()[:28]})
df = pd.DataFrame(rows)
if df.empty:
    print("API 호출이 모두 실패해 표를 만들지 못했습니다. 위 안내(429·401)를 확인하고 잠시 뒤 이 셀을 다시 실행하세요.")
else:
    display(df)
    print(f"정답률 {df['정답'].eq('O').mean():.0%} · 캐시 적중 평균 {df['캐시'].mean():.0f} / 입력 {df['입력'].mean():.0f} 토큰")
    print(f"{len(df)}건 비용: 캐시 적용 ${usd_with:.6f}  vs  캐시 없음 ${usd_without:.6f}  (약 {usd_with/usd_without:.0%}, 첫 호출의 캐시 쓰기 1.25배 포함)")

💡 **해석** — 첫 호출(또는 9절 직후라면 모든 호출)부터 접두부 약 2,500토큰이 캐시에서 옵니다. 질문 50토큰만 정가를 냅니다. 10건이면 작은 돈이지만 **하루 1만 건 평가·운영**이면 입력 비용이 약 10분의 1로 줄어듭니다. 정답률은 캐시와 무관합니다. 캐시는 **계산 결과를 재사용**하는 것이지 답을 바꾸지 않습니다. 반대로 접두부를 바꾸는 작은 변경(날짜를 지시문 맨 앞에 넣기, 규정집 문장 하나 수정, 모델 변경)은 캐시를 전부 깨뜨립니다. 그래서 평가 루프에서는 **접두부를 고정하고 질문만 바꿉니다.**

### 9-3. 누적 대화: messages는 append되어 매번 전부 다시 간다 🔵
API는 이전 호출을 기억하지 않습니다. 대화를 이어가려면 **클라이언트가 지금까지의 메시지를 전부 다시 보냅니다.** 그래서 턴이 늘수록 `input_tokens`가 커지고, 앞부분(지시문 + 지난 대화)이 같으므로 캐시가 맞습니다. ChatGPT 화면이 앞 대화를 기억하는 것처럼 보이는 것도 앱이 기록을 다시 보내기 때문입니다.

In [ ]:
history = []        # 지금까지의 대화: user·assistant 메시지를 순서대로 누적
turns = ["과장의 국내 출장 일비는?", "그럼 차장은?", "본인 결혼 경조금은?", "어학 교육비는 월 얼마까지?"]
for i, q in enumerate(turns, 1):
    history.append({"role": "user", "content": q})
    r = ask(history, instructions=RULES_INST, max_output_tokens=80, prompt_cache_key="hanbit-rules-v1")   # input=리스트 전체
    if not r: break
    history.append({"role": "assistant", "content": r.output_text})
    print(f"턴 {i} · 보낸 메시지 {len(history)-1}개 · A: {r.output_text.strip()[:50]}\n")

💡 **해석** — 턴 2의 "그럼 차장은?"이 통하는 이유는 턴 1의 질문과 답이 `history`에 실려 함께 갔기 때문입니다. 입력 토큰은 턴마다 늘어납니다(지시문 + 지난 대화 전부 + 새 질문). 이 모델의 컨텍스트 윈도우는 1,050,000토큰이지만, 거기에 닿으면 기본 설정(`truncation="disabled"`)은 **400 오류**를 내고 과거를 알아서 지우지 않습니다. `truncation="auto"`를 주면 서버가 **대화의 앞부분부터 버려** 맞춥니다(공식 문서 문구). 긴 대화를 운영하려면 아래 세 가지 보완이 필요합니다.

In [ ]:
probe = {"role": "user", "content": "맨 처음에 물어본 직급이 뭐였지?"}

# ① 슬라이딩 윈도우: 지시문은 유지하고 최근 N개 메시지만 보낸다 (오래된 턴은 잊는다 · 접두부가 바뀌므로 캐시는 깨진다)
window = history[-2:]
r = ask(window + [probe], instructions=RULES_INST, max_output_tokens=60)
if r: print("① 윈도우(최근 2개)만:", r.output_text.strip()[:60])

# ② 요약 압축: 오래된 턴을 요약 한 덩어리로 바꾼다 (사실은 남기고 토큰은 줄인다)
s = ask("다음 대화를 질문·답 중심으로 두 문장으로 요약하라:\n" + json.dumps(history, ensure_ascii=False), max_output_tokens=120)
if s:
    compressed = [{"role": "user", "content": "[이전 대화 요약] " + s.output_text.strip()}]
    r = ask(compressed + [probe], instructions=RULES_INST, max_output_tokens=60)
    if r: print("② 요약 후:", r.output_text.strip()[:60])

# ③ 서버가 기억: previous_response_id 로 잇는다 (store=True 기본, 30일 보관). instructions는 이어지지 않으므로 매번 다시 보낸다.
r1 = ask("과장의 국내 출장 일비는?", instructions=RULES_INST, max_output_tokens=60)
if r1:
    r2 = ask("그럼 차장은?", instructions=RULES_INST, max_output_tokens=60, previous_response_id=r1.id)
    if r2: print("③ previous_response_id:", r2.output_text.strip()[:60], "· 입력 토큰에 이전 턴이 포함된 것을 확인")

💡 **해석** — ①은 싸지만 첫 질문을 잊습니다. ②는 요약 호출 비용이 한 번 더 들지만 핵심 사실은 남습니다. ③은 서버가 이전 응답을 보관하므로 코드가 단순하지만, 이전 턴의 토큰은 여전히 과금되고 `instructions`는 매 요청에 다시 넣어야 합니다. 실무에서는 "최근 N턴은 원문, 그 이전은 요약" 조합이 흔합니다. 어떤 방식이든 **접두부를 고정**해야 9절의 캐시 할인이 유지됩니다.

## 10. 바꿔서 실행 🟡
들어온 문의 5건을 한 번에 분류해 표로 만듭니다. 문의를 바꾸거나 늘려 보고, `Inquiry`에 필드(예: `department: Literal["인사팀","총무팀","재무팀"]`)를 추가해 보세요.

In [ ]:
import pandas as pd
inquiries = [   # ← 바꿔 보세요
    "다음 달 싱가포르 출장인데 숙박비 상한이 얼마인지 알려 주세요.",
    "아버지 칠순 잔치가 있는데 경조사 휴가가 나오나요?",
    "AWS 자격증 시험 응시료도 교육비 지원이 되나요?",
    "회사 주차장 정기권은 어디서 신청하나요?",
    "오늘 밤 비행기로 급히 일본 출장 갑니다. 사후 승인 절차가 뭐죠?",
]
rows = []
for text in inquiries:
    res = classify(text)
    rows.append({"문의": text[:28] + "…", **(res.model_dump() if res else {"category": None, "urgency": None, "summary": "(API 실패)"})})
df = pd.DataFrame(rows)
display(df)

### 🤔 생각해보기

규정 도우미가 **하루 1만 건**의 질문을 받는다고 합시다. 매번 규정 전체(약 2,500토큰)를 지시문으로 보내고 질문 50토큰, 답 150토큰이라면 한 달(30일) 비용은? 캐시가 90% 적중하면 얼마로 줄까요? 아래 코드로 직접 계산해 보세요.

```python
n = 10_000 * 30
print(cost_usd('gpt-6-luna', 2550, 150) * n)                    # 캐시 없음
print(cost_usd('gpt-6-luna', 2550, 150, cached_tok=int(2500*0.9)) * n)   # 캐시 90%
```

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- 캐시 없음: (2,550 × $0.10 + 150 × $0.50) / 1M × 30만 건 = (0.000255 + 0.000075) × 30만 ≈ **$99/월**(약 14만 원).
- 캐시 90% 적중: 입력 중 2,250토큰이 $0.01 단가 → (300 × 0.10 + 2,250 × 0.01 + 150 × 0.50) / 1M = $0.0001275 × 30만 ≈ **$38/월**.
- 입력 비용만 보면 약 4분의 1로 줄고, 이제 **출력 비용이 절반 이상**을 차지합니다. 다음 절감 수단은 `max_output_tokens`와 짧은 답 지시입니다.
- 모델을 `gpt-6-astra`(입력 $10)로 바꾸면 100배가 됩니다. 쉬운 질문은 작은 모델, 어려운 질문만 큰 모델로 보내는 **라우팅**이 여기서 나옵니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 새 프로젝트는 Responses API. 결과는 `output_text`, 과금은 `usage`.
- `instructions`는 규칙, `input`은 질문. 고정 지시문을 앞에 두면 캐시를 받는다. 평가·배치처럼 같은 접두부를 반복하는 일에서 입력 비용이 10분의 1이 된다.
- 대화는 클라이언트가 messages를 append해 매번 전부 보낸다. 한도를 넘으면 400 오류. 보완은 슬라이딩 윈도우·요약·previous_response_id.
- effort는 기본 none, 틀리는 문제만 올린다. 추론 토큰도 출력으로 과금된다.
- 형식은 `responses.parse` + Pydantic으로 강제한다. 내용 정확성은 별도로 평가한다.
- 401·일반 429·지출 한도 429는 처리 방법이 다르다.

> 다음 노트북: **13_tool_calling_open_vs_api — 모델에게 도구라는 손 달아 주기**